# Projet n°1 : Scrapping

Date : october 2025

Author : Arsena Julien

In [ ]:
import requests
import time
import random

max_pages = 60
dataset = "stzhao/movie_posters_100k_controlnet"
split = "train"
config = "default"

headers = {"User-Agent": "Mozilla/5.0"}

soup_title = []
soup_image = []
soup_year = []
soup_genre = []
soup_budget = []
soup_lang = []

for page_index in range(max_pages):
    offset_value = page_index * 100
    url = f"https://datasets-server.huggingface.co/rows?dataset={dataset}&config={config}&split={split}&offset={offset_value}&length=50"
    page = requests.get(url, headers=headers, timeout=50)
    if page.status_code != 200 or "application/json" not in page.headers.get("content-type",""):
        print(f"[Skip page {page_index+1}] HTTP={page.status_code}")
        time.sleep(random.uniform(0.8, 1.5))
        continue

    data = page.json()
    rows = data.get("rows")
    if not rows:
        print(f"[Fin] plus de lignes à offset {offset_value}")
        break

    for item in rows:
        row = item.get("row", {})

        title = row.get("title", "Inconnu")

        # image url
        img_obj = row.get("image", {})
        image_url = img_obj.get("src", "Non spécifié") if isinstance(img_obj, dict) else "Non spécifié"

        # year from release_date
        release_date = row.get("release_date") or row.get("releaseDate")
        year = release_date[:4] if isinstance(release_date, str) and len(release_date) >= 4 else "Non spécifié"

        # genres (list -> string)
        genres = row.get("genres") or row.get("genre")
        if isinstance(genres, list):
            genre = "|".join([str(g) for g in genres])
        else:
            genre = genres if isinstance(genres, str) else "Non spécifié"

        # budget and original language
        budget = row.get("budget", "Non spécifié")
        orig_lang = row.get("original_language") or row.get("language") or "Non spécifié"

        soup_title.append(title)
        soup_image.append(image_url)
        soup_year.append(year)
        soup_genre.append(genre)
        soup_budget.append(budget)
        soup_lang.append(orig_lang)

        time.sleep(random.uniform(0.2, 0.5))

    print(f"Page {page_index+1} traitée.")

print("Exemple :", soup_title[:5], soup_year[:5], soup_genre[:5], soup_budget[:5], soup_lang[:5], soup_image[:2])


Page 1 traitée.
Page 2 traitée.
Page 3 traitée.
Page 4 traitée.
Page 5 traitée.
Page 6 traitée.
Page 7 traitée.
Page 8 traitée.
Page 9 traitée.
Page 10 traitée.
Page 11 traitée.
Page 12 traitée.
Page 13 traitée.
Page 14 traitée.
Page 15 traitée.
Page 16 traitée.
Page 17 traitée.
Page 18 traitée.
Page 19 traitée.
Page 20 traitée.
Page 21 traitée.
Page 22 traitée.
Page 23 traitée.
Page 24 traitée.
Page 25 traitée.
Page 26 traitée.
Page 27 traitée.
Page 28 traitée.
Page 29 traitée.
Page 30 traitée.
Page 31 traitée.
Page 32 traitée.
Page 33 traitée.
Page 34 traitée.
Page 35 traitée.
Page 36 traitée.
Page 37 traitée.
Page 38 traitée.
Page 39 traitée.
Page 40 traitée.
Page 41 traitée.
Page 42 traitée.
Page 43 traitée.
Page 44 traitée.
Page 45 traitée.
Page 46 traitée.
Page 47 traitée.
Page 48 traitée.
Page 49 traitée.
Page 50 traitée.
Page 51 traitée.
Page 52 traitée.
Page 53 traitée.
Page 54 traitée.
Page 55 traitée.
Page 56 traitée.
Page 57 traitée.
Page 58 traitée.
Page 59 traitée.
Page 6

In [2]:
import pandas as pd
df = pd.DataFrame({'title' : soup_title, 
                   'year' : soup_year, 
                   'genre' : soup_genre, 
                   'image' : soup_image,
                   'budget' : soup_budget,
                   'original_language' : soup_lang})

In [3]:
df

,title,year,genre,image,budget,original_language
0,57 Seconds,2023,"{'id': 53, 'name': 'Thriller'}|{'id': 878, 'na...",https://datasets-server.huggingface.co/cached-...,0,en
1,Mission: Impossible - Dead Reckoning Part One,2023,"{'id': 28, 'name': 'Action'}|{'id': 53, 'name'...",https://datasets-server.huggingface.co/cached-...,291000000,en
2,Expend4bles,2023,"{'id': 28, 'name': 'Action'}|{'id': 12, 'name'...",https://datasets-server.huggingface.co/cached-...,100000000,en
3,Sound of Freedom,2023,"{'id': 28, 'name': 'Action'}|{'id': 18, 'name'...",https://datasets-server.huggingface.co/cached-...,15000000,en
4,Nowhere,2023,"{'id': 53, 'name': 'Thriller'}|{'id': 18, 'nam...",https://datasets-server.huggingface.co/cached-...,0,es
...,...,...,...,...,...,...
2995,Mom,2017,"{'id': 80, 'name': 'Crime'}|{'id': 18, 'name':...",https://datasets-server.huggingface.co/cached-...,0,hi
2996,In the Morning of La Petite Mort,2023,"{'id': 18, 'name': 'Drama'}",https://datasets-server.huggingface.co/cached-...,0,zh
2997,Deadstream,2022,"{'id': 27, 'name': 'Horror'}|{'id': 35, 'name'...",https://datasets-server.huggingface.co/cached-...,0,en
2998,Sansón and Me,2023,"{'id': 99, 'name': 'Documentary'}",https://datasets-server.huggingface.co/cached-...,0,es


In [4]:
df.to_csv('projetn1.csv')

In [ ]:
import io, time, random, requests, numpy as np, pandas as pd, colorsys
from PIL import Image
from sklearn.cluster import KMeans
from tqdm import tqdm

TOP_K = 5
K_CLUSTERS = 5
MAX_THUMB = (256, 256)
TIMEOUT = 15
HEADERS = {"User-Agent": "Mozilla/5.0"}
INCLUDE_HEX = True

df = pd.DataFrame({
    "title": soup_title,
    "image_url": soup_image,
    "year": soup_year,
    "genre": soup_genre,
    "budget": soup_budget,
    "original_language": soup_lang
})

# result columns
for i in range(1, TOP_K+1):
    df[f"couleur_{i}"] = ""
    if INCLUDE_HEX:
        df[f"hex_{i}"] = ""

mask_valid = df["image_url"].astype(str).str.startswith("http", na=False)
df_valid = df[mask_valid].copy()

ok, skip_bad_url, open_fail, kmeans_fail = 0, 0, 0, 0
print(f"URLs valides détectées: {len(df_valid)} / {len(df)}")

for row in tqdm(df_valid.itertuples(), total=len(df_valid)):
    url = row.image_url
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        if r.status_code != 200:
            skip_bad_url += 1
            continue
        # Try to open even if the header doesn't indicate 'image'
        try:
            img = Image.open(io.BytesIO(r.content)).convert("RGB")
        except Exception:
            open_fail += 1
            continue
        img.thumbnail(MAX_THUMB, Image.LANCZOS)
        arr = np.asarray(img, dtype=np.uint8).reshape(-1, 3)
        if arr.shape[0] > 200_000:
            arr = arr[np.random.choice(arr.shape[0], 200_000, replace=False)]

        km = KMeans(n_clusters=K_CLUSTERS, random_state=42, n_init='auto')
        labels = km.fit_predict(arr)
        centers = km.cluster_centers_.astype(int)
        counts = np.bincount(labels, minlength=K_CLUSTERS).astype(float)
        centers = centers[np.argsort(-counts)]

        for j in range(TOP_K):
            r_, g_, b_ = [int(x) for x in centers[j]]
            h, s, v = colorsys.rgb_to_hsv(r_/255, g_/255, b_/255); h = (h*360)%360
            # h in [0,360), s,v in [0,1]
            # Prioritize B&W/gray handling
            if v > 0.92 and s < 0.12:
                name = "blanc"
            elif v < 0.08:
                name = "noir"
            elif s < 0.18:
                name = "gris clair" if v >= 0.70 else ("gris foncé" if v <= 0.35 else "gris")
            else:
                # Finer hue bins to better separate red/pink/magenta/violet
                if (h >= 345 or h < 15):          base = "rouge"     # core red
                elif h < 35:                      base = "orange"
                elif h < 65:                      base = "jaune"
                elif h < 170:                     base = "vert"
                elif h < 200:                     base = "turquoise"
                elif h < 255:                     base = "bleu"
                elif h < 285:                     base = "violet"    # blue-violet
                elif h < 315:                     base = "magenta"   # saturated violet-red
                elif h < 345:                     base = "rose"      # distinct pink zone
                else:                             base = "rouge"

                # Correction rules around red:
                # - "marron" = dark orange (brown)
                if base == "orange" and v < 0.50 and s > 0.35:
                    base = "marron"

                # - "rose" if red/magenta but light and not too saturated
                if base in {"rouge", "magenta"} and v > 0.75 and s < 0.60:
                    base = "rose"

                # - "magenta" if red-violet area very saturated and not too dark
                if base == "rouge" and 300 <= h < 345 and s >= 0.60 and v >= 0.40:
                    base = "magenta"

                # Light/dark modifiers (not for white/black/gray/brown)
                if base not in {"blanc", "noir", "gris", "marron"}:
                    if v > 0.78 and s > 0.22:
                        base += " clair"
                    elif v < 0.38 and s > 0.22:
                        base += " foncé"

                name = base

            df.loc[row.Index, f"couleur_{j+1}"] = name
            if INCLUDE_HEX:
                df.loc[row.Index, f"hex_{j+1}"] = f"#{r_:02x}{g_:02x}{b_:02x}"
        ok += 1
    except Exception:
        kmeans_fail += 1
        continue
    time.sleep(random.uniform(0.03, 0.1))

print(f"OK: {ok} | HTTP non 200: {skip_bad_url} | Ouverture image échouée: {open_fail} | Échec KMeans/autre: {kmeans_fail}")

# Visual check
display_cols = ["title"] + [f"couleur_{i}" for i in range(1, TOP_K+1)]
print(df[display_cols].head(10))


URLs valides détectées: 3000 / 3000


100%|██████████| 3000/3000 [47:47<00:00,  1.05it/s]   


OK: 2992 | HTTP non 200: 0 | Ouverture image échouée: 0 | Échec KMeans/autre: 8
                                           title    couleur_1  \
0                                     57 Seconds        blanc   
1  Mission: Impossible - Dead Reckoning Part One        blanc   
2                                    Expend4bles  rouge foncé   
3                               Sound of Freedom        blanc   
4                                        Nowhere   gris foncé   
5                                     The Nun II         noir   
6                                     Talk to Me         noir   
7                                   Gran Turismo         noir   
8                                      Ballerina   rose foncé   
9                                    Retribution   bleu foncé   

         couleur_2    couleur_3    couleur_4   couleur_5  
0       gris foncé         bleu         gris  gris clair  
1       gris foncé        rouge   gris clair        bleu  
2            rouge  rouge f

In [6]:
df

,title,image_url,year,genre,budget,original_language,couleur_1,hex_1,couleur_2,hex_2,couleur_3,hex_3,couleur_4,hex_4,couleur_5,hex_5
0,57 Seconds,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 53, 'name': 'Thriller'}|{'id': 878, 'na...",0,en,blanc,#f1eee9,gris foncé,#23242a,bleu,#3f474f,gris,#737b7e,gris clair,#c0b6ae
1,Mission: Impossible - Dead Reckoning Part One,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 28, 'name': 'Action'}|{'id': 53, 'name'...",291000000,en,blanc,#f8f6f0,gris foncé,#2d292d,rouge,#a35f50,gris clair,#bdb0ae,bleu,#586f8a
2,Expend4bles,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 28, 'name': 'Action'}|{'id': 12, 'name'...",100000000,en,rouge foncé,#150706,rouge,#ad231c,rouge foncé,#5e1c1c,jaune clair,#d2962e,vert clair,#5fcd7f
3,Sound of Freedom,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 28, 'name': 'Action'}|{'id': 18, 'name'...",15000000,en,blanc,#fef6ee,noir,#0e0d0f,gris foncé,#3f3935,gris,#7c726a,gris clair,#bdb5ab
4,Nowhere,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 53, 'name': 'Thriller'}|{'id': 18, 'nam...",0,es,gris foncé,#293130,noir,#0e0f0f,gris foncé,#495451,gris,#808373,jaune,#bdad8e
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,Mom,https://datasets-server.huggingface.co/cached-...,2017,"{'id': 80, 'name': 'Crime'}|{'id': 18, 'name':...",0,hi,turquoise foncé,#102129,turquoise,#364345,gris,#888076,gris clair,#d7d6d2,rouge,#bf383e
2996,In the Morning of La Petite Mort,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 18, 'name': 'Drama'}",0,zh,rose foncé,#22181d,rose,#504147,gris,#afa3a5,gris,#7d7276,gris clair,#d5d0d6
2997,Deadstream,https://datasets-server.huggingface.co/cached-...,2022,"{'id': 27, 'name': 'Horror'}|{'id': 35, 'name'...",0,en,noir,#100703,marron,#291b10,marron,#56412e,jaune,#b09a5c,orange,#b24017
2998,Sansón and Me,https://datasets-server.huggingface.co/cached-...,2023,"{'id': 99, 'name': 'Documentary'}",0,es,turquoise,#3e7171,noir,#090508,rose foncé,#462d36,rose clair,#d11392,orange,#a67257


In [7]:
df.to_csv('projetn1.csv')